## DLT Notebook Gold layers

DLT is a declarative ETL framework. Focus on "what step" others DLT take cares


1. Streaming table - table similar to created in autoloader
2. materialized view - holds query + data
3. Views - holds query (types: normal view, streaming view)

Data quality/expactations in DLT: <br>
any stream, view/ materialized view create. We set rules some data quality checks, data constraints. if they fail<br>
1. Warn - warn and feed data into target location (continue processing)
2. Fail - Simply fail the flow
3. Drop - It will just pass data that is passed to the continue processing(target) failed are dropped

dictionary used as a configuration for data quality.

Instead of writing the same rule 10 times for 10 different tables, store it here.<br>
It tells DLT: "Every row must have a show_id or it's considered bad data."

In [0]:
looktables_rules ={
    "first_rule":" show_id is NOT NULL"
}

In [0]:
import dlt

1. expectAll -
2. expect_or_drop-
3. expect_or_fail -
4. expect - Default is warn

decoratorer @dlt - A decorator that tells Databricks to create a managed table in  catalog.             


In [0]:
@dlt.table(
  name = "gold_netflixdirectors"
)

@dlt.expect_all_or_drop(looktables_rules)
def gold_directors():
  df = spark.readStream.format("delta").load("abfss://silver@netflixprojectdlsubhik.dfs.core.windows.net/netflix_directors")
  return df

In [0]:
@dlt.table(
  name = "gold_netflixcast"
)

@dlt.expect_all_or_drop(looktables_rules)
def gold_cast():
  df = spark.readStream.format("delta").load("abfss://silver@netflixprojectdlsubhik.dfs.core.windows.net/netflix_cast")
  return df

In [0]:
@dlt.table(
  name = "gold_netflixcountries"
)

@dlt.expect_all_or_drop(looktables_rules)
def gold_countries():
  df = spark.readStream.format("delta").load("abfss://silver@netflixprojectdlsubhik.dfs.core.windows.net/netflix_countries")
  return df

In [0]:
@dlt.table(
  name = "gold_netflixcategory"
)

@dlt.expect_or_drop("first_rule", "show_id is NOT NULL")
def gold_category():
  df = spark.readStream.format("delta").load("abfss://silver@netflixprojectdlsubhik.dfs.core.windows.net/netflix_category")
  return df

1 common column in all table is Show_id this would be called as expectation. in cell 4 dict is created

Staging layer

In [0]:
@dlt.table

def gold_stg_netflixtitles():
    df = spark.readStream.format("delta").load("abfss://silver@netflixprojectdlsubhik.dfs.core.windows.net/netflix_titles")
    return df

In [0]:
from pyspark.sql.functions import * 

In [0]:
@dlt.view

def gold_transform_netflixtitles():
  df = spark.readStream.table("LIVE.gold_stg_netflixtitles")
  df = df.withColumn("newflag", lit(1))
  return df

In [0]:
masterdata_rules = {
    "first_rule":" newflag is NOT NULL",
    "second_rule":" show_id is NOT NULL"
}

In [0]:
@dlt.table

@dlt.expect_all_or_drop(masterdata_rules)
def gold_netflixtitles_final():
  df = spark.readStream.table("LIVE.gold_transform_netflixtitles")
  return df

1. Aggregations (Summarizing Data)
This creates a table that business users can use to quickly see which countries produce the most content.

In [0]:
@dlt.table(
    name="gold_content_count_by_country",
    comment="Summary of total titles per country"
)
def gold_content_summary():
    # Reading from your Silver table
    return (
        spark.readStream.table("LIVE.gold_netflixcountries")
        .groupBy("country")
        .count()
        .withColumnRenamed("count", "total_titles")
    )

2. Joining (Creating a "Wide" Master Table)
Gold layers often join multiple Silver tables so that a BI tool doesn't have to do the heavy lifting later.

In [0]:
@dlt.table(
    name="gold_master_netflix_details",
    comment="Joined table containing titles and their primary categories"
)
def gold_master_data():
    df_titles = spark.readStream.table("LIVE.gold_netflixtitles_final")
    df_category = spark.readStream.table("LIVE.gold_netflixcategory")
    
    # Joining on show_id
    return df_titles.join(df_category, "show_id", "inner")

3. Business Logic (Filtering & Labelling)
You can categorize data based on specific business rules (e.g., separating "Old" vs "New" content).

In [0]:
from pyspark.sql.functions import col, when

@dlt.table(
    name="gold_recent_movie_highlights"
)
def gold_recent_movies():
    return (
        spark.readStream.table("LIVE.gold_netflixtitles_final")
        .filter(col("type") == "Movie")
        .withColumn("release_era", 
            when(col("release_year") >= 2020, "Modern")
            .otherwise("Classic")
        )
    )

4. Advanced Data Quality (Quarantine Table)
While you use expect_or_drop to keep Gold clean, you might want a separate table to see exactly what was dropped for auditing.

In [0]:
@dlt.table(
    name="gold_invalid_records_audit"
)
# We use a 'warn' expectation here so the bad data actually enters THIS specific table
@dlt.expect("valid_show_id", "show_id IS NOT NULL") 
def gold_audit_table():
    return (
        spark.readStream.format("delta")
        .load("abfss://silver@netflixprojectdlsubhik.dfs.core.windows.net/netflix_titles")
        .filter(col("show_id").isNull())
    )